## Import Dataset

In [63]:
import pandas as pd
df = pd.read_csv("../../raw_data/recipes_ingredients.csv")

## Checking stuff

In [109]:
df.shape

(500471, 9)

In [110]:
df.head()

,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags
0,71247,Cherry Streusel Cobbler,"I haven't made this in years, so I'm just gues...","[""cherry pie filling"", ""condensed milk"", ""melt...","[""2 (21 ounce) cans cherry pie filling"",""2...","[""Preheat oven to 375°F."", ""Spread cherry pie ...",6.0,1 (347 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
1,76133,Reuben and Swiss Casserole Bake,I think this is even better than a reuben sand...,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
2,503816,Yam-Pecan Recipe,A lady I work with heard me taking about ZWT a...,"[""unsalted butter"", ""vegetable oil"", ""all - pu...","[""3/4 cup unsalted butter, at room tempera...","[""Preheat oven to 350°F In a mixing bowl, usi...",8.0,1 (198 g),"[""time-to-make"", ""course"", ""main-ingredient"", ..."
3,418749,Tropical Orange Layer Cake,An easy and delicious cake. Great for a summ...,"[""orange cake mix"", ""instant vanilla pudding"",...","[""1 (18 ounce) pkge.orange cake mix"",""1 (3...","[""In a large mixing bowl, combine the first 6 ...",16.0,1 (191 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
4,392934,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,I was searching the web for something like thi...,"[""butter"", ""brown sugar"", ""granulated sugar"", ...","[""1/2 cup butter, room temperature "",""1/2 ...","[""Cream butter and sugars together."", ""Blend i...",24.0,1 (26 g),"[""15-minutes-or-less"", ""time-to-make"", ""course..."


## Preprocessing

In [112]:
data = df.dropna()  #enlève les NaN

In [ ]:
data = data.drop(columns=["description","id"])  # on enlève les colonnes innutiles
data.head()

,name,ingredients,ingredients_raw,steps,servings,serving_size,tags
0,Cherry Streusel Cobbler,"[""cherry pie filling"", ""condensed milk"", ""melt...","[""2 (21 ounce) cans cherry pie filling"",""2...","[""Preheat oven to 375°F."", ""Spread cherry pie ...",6.0,1 (347 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
1,Reuben and Swiss Casserole Bake,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
2,Yam-Pecan Recipe,"[""unsalted butter"", ""vegetable oil"", ""all - pu...","[""3/4 cup unsalted butter, at room tempera...","[""Preheat oven to 350°F In a mixing bowl, usi...",8.0,1 (198 g),"[""time-to-make"", ""course"", ""main-ingredient"", ..."
3,Tropical Orange Layer Cake,"[""orange cake mix"", ""instant vanilla pudding"",...","[""1 (18 ounce) pkge.orange cake mix"",""1 (3...","[""In a large mixing bowl, combine the first 6 ...",16.0,1 (191 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
4,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,"[""butter"", ""brown sugar"", ""granulated sugar"", ...","[""1/2 cup butter, room temperature "",""1/2 ...","[""Cream butter and sugars together."", ""Blend i...",24.0,1 (26 g),"[""15-minutes-or-less"", ""time-to-make"", ""course..."


## Fonction pour transformer les strings en liste.
## on applique sur les colonnes concernés (ingredients, ingredients_raw, steps, tags)

In [114]:
import ast

errors = []

def safe_literal_eval(value):
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError) as e:
        errors.append((value, str(e)))
        return value

data["ingredients"] = data["ingredients"].apply(safe_literal_eval)
data["ingredients_raw"] = data["ingredients_raw"].apply(safe_literal_eval)
data["steps"] = data["steps"].apply(safe_literal_eval)
data["tags"] = data["tags"].apply(safe_literal_eval)


In [ ]:
type(data.iloc[0].steps) #vérification

list

In [ ]:
data.shape #shape avant

(497563, 7)

## On enlève les lignes sans ingrédients, ingredients_raw, steps et tags

In [124]:
data = data[data["ingredients"].apply(len) > 0]
data = data[data["ingredients_raw"].apply(len) > 0]
data = data[data["steps"].apply(len) > 0]
data = data[data["tags"].apply(len) > 0]

In [125]:
data.shape

(489757, 7)

## On teste avec une liste d'ingrédient fictive

In [131]:
user_ingredients = ["tomato", "chicken", "onion", "garlic", "apple", "sugar", "cheese", "bread"]
top_recipes = data.copy() # New Dataset pour rank en fonction de la liste

# Fonction pour compter le nombre de matchs entre la liste et le dataset

In [127]:
def count_matches(recipe_ingredients, user_ingredients):
    matches = 0

    for user_ing in user_ingredients:
        for recipe_ing in recipe_ingredients:
            if user_ing.lower() in recipe_ing.lower():
                matches += 1
                break

    return matches

# Ajoute au nouveau dataset , une colonne match count pour compter le nombre de matchs

In [132]:
top_recipes["match_count"] = top_recipes["ingredients"].apply(lambda x: count_matches(x, user_ingredients))

In [135]:
top_recipes.sort_values("match_count", ascending=False, inplace=True)

In [136]:
top_recipes.head()

,name,ingredients,ingredients_raw,steps,servings,serving_size,tags,match_count
234291,BBQ Chicken Packed Pita,[boneless skinless chicken thighs breasts thig...,[1 lb boneless skinless chicken thighs (o...,[Prepare a hot grill outside or preheat a gril...,1.0,1 (4047 g),"[weeknight, 60-minutes-or-less, time-to-make, ...",8
68752,Manchu Spiced Garlic Chicken Pizzettas,"[olive oil, olive oil, large onions, golden br...","[1/4 cup olive oil, plus , 2 tablespoons...",[Heat 1/4 cup oil in heavy large skillet over ...,1.0,1 (1363 g),"[60-minutes-or-less, time-to-make, course, mai...",7
140096,Ultimate Chicken Parmigiana,"[virgin olive oil, virgin olive oil, medium on...","[1/4 cup extra virgin olive oil, plus , 3 ...","[Preheat the oven to 350 degrees F., Coat a sa...",4.0,1 (864 g),"[time-to-make, course, main-ingredient, cuisin...",7
442260,Memphis Chicken With Coleslaw and a Potato Stack,"[chicken breast fillets, smoked paprika, dried...","[6 chicken breast fillets, 1 teaspoon ...",[In a large bowl mix together the chicken fill...,6.0,1 (464 g),"[time-to-make, course, main-ingredient, cuisin...",7
95484,Delicious Pressure Cooker Chili,"[ground beef, medium onion, garlic cloves minc...","[1 1/2 lbs ground beef, 1 medium onion, diced,...",[Heat up instant pot using Saute More function...,46.0,1 (543 g),"[60-minutes-or-less, time-to-make, course, pre...",7


# vérification des ingrédients à la main

In [108]:
liste = top_recipes.iloc[1].ingredients
liste

['chicken breast',
 'soy sauce',
 'lime juice',
 'canola oil',
 'garlic minced',
 'brown sugar',
 'cumin',
 'chili powder',
 'cheddar cheese shredded',
 'onions',
 'salt pepper',
 'tomatoes beefsteak',
 'iceberg lettuce',
 'pita bread']